# 07 — Iterate

Data-driven component replication.

**What you learn:**
- `iterate='^path'`: replicate a component once per child in data
- `^.?attr`: resolve an attribute from the current data node
- Each iteration gets its own data namespace
- Data is a `Bag` of nodes with attributes (`set_item` with kwargs)

**Prerequisites:** 05_components, 04_pointers

In [ ]:
from IPython.display import HTML
from genro_bag import Bag
from genro_builders.builder import component
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.manager import BuilderManager


class CatalogBuilder(HtmlBuilder):
    @component(main_tag='div', sub_tags='')
    def product_card(self, comp, **kwargs):
        """One card per data node. ^.?name reads from the current node's attrs."""
        card = comp.div(_class='card')
        card.h3('^.?name')
        card.p('^.?price', _class='price')
        card.p('^.?description')
        card.p('^.?category', _class='category')

In [ ]:
class ProductCatalog(BuilderManager):
    def __init__(self):
        self.page = self.set_builder('page', CatalogBuilder)
        self.run()

    def store(self, data):
        products = Bag()
        products.set_item('p0', None, name='Wireless Headphones', price='$79.99',
                          description='Premium sound, 30h battery.', category='Audio')
        products.set_item('p1', None, name='Mechanical Keyboard', price='$129.99',
                          description='Cherry MX, RGB backlight.', category='Input')
        products.set_item('p2', None, name='USB-C Hub', price='$49.99',
                          description='7-in-1: HDMI, USB-A, SD, ethernet.', category='Accessories')
        products.set_item('p3', None, name='4K Monitor', price='$349.99',
                          description='27-inch IPS, 144Hz, USB-C.', category='Display')
        data['products'] = products

    def main(self, source):
        source.head().style('''
            body { font-family: sans-serif; max-width: 700px; margin: 0 auto; }
            .catalog { display: grid; grid-template-columns: repeat(auto-fill, minmax(200px, 1fr)); gap: 1em; }
            .card { border: 1px solid #ddd; border-radius: 8px; padding: 1em; }
            .card h3 { margin-top: 0; }
            .price { color: #16a34a; font-weight: bold; font-size: 1.2em; }
            .category { color: #666; font-size: 0.85em; text-transform: uppercase; }
        ''')
        body = source.body()
        body.h1('Product Catalog')
        body.p('Each card generated by iterate — one per product.')
        catalog = body.div(_class='catalog')
        catalog.product_card(iterate='^products')

app = ProductCatalog()
HTML(app.page.render())